# linspace-out-param — worked example 3: out= preserves pointer, fresh linspace does not

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linspace-out-param`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`torch.linspace(..., out=buf)` writes into existing storage so `buf.data_ptr()` is preserved. Plain `buf = torch.linspace(...)` rebinds the name to a freshly allocated tensor with a new pointer. Both yield identical values; only the memory behavior differs.

## Worked solution

We compare the two idioms on equal-shaped buffers.

1. **Record pointers.** Save `data_ptr()` for both buffers before any fill.
2. **out= idiom.** `torch.linspace(start, end, N, out=buf_a)` writes into `buf_a`; its pointer is unchanged afterward.
3. **Rebind idiom.** `buf_b = torch.linspace(start, end, N)` allocates a new tensor and rebinds the local name; the original `buf_b` storage is abandoned, so the new pointer differs from the recorded one.
4. **Same values.** `torch.equal(buf_a, buf_b)` is True — the numbers match; the lesson is purely about allocation.

The demo prints whether each pointer was preserved and whether the values match.

In [ ]:
import torch as t

t.manual_seed(2)
N = 7
buf_a = t.zeros(N)
buf_b = t.zeros(N)
ptr_a0 = buf_a.data_ptr()
ptr_b0 = buf_b.data_ptr()

t.linspace(1.0, 2.0, N, out=buf_a)        # in-place: pointer preserved
buf_b = t.linspace(1.0, 2.0, N)            # rebind: new allocation

print('a pointer preserved:', buf_a.data_ptr() == ptr_a0)
print('b pointer preserved:', buf_b.data_ptr() == ptr_b0)
print('values equal:', bool(t.equal(buf_a, buf_b)))